In [3]:
### Description of the code:
# The analysis is run in Python (version 3.10.14) in Jupyter Notebook. 
# The script first opens a .czi file of the acquired dimension of several tiles, a z-stack of 80 slices, 2 to 4 colors, and 512x512 pixels. 
# Extract and convert to image slices, then enhance the contrast per slice with Contrast Limited Adaptive Histogram Equalization (CLAHE) so edges are sharpened. 
# Subsequently adaptive thresholding creates a binary mask to extract potential GUVs. 
# MorphologyEx function removes noise and small dots so that a distance transform can be applied and local maxima in the distance map can be detected to apply the watershed. 
# The peaks (local maxima) that presumably belong to one GUVs each are labeled with an integer number. 
# Those masks are tested on geometric parameters and too small, too large, too eccentric or touching the image border ones are filtered out. 
# The global GUV center is determined to be the stack where the radius is largest, which is determined across multiple slices of the z-stack, and which is then stored as the final GUV center.
# Overlay images with the drawn detected GUV circumference on the real image data are created for clarity. 
# Slice-by-slice images are created and saved.


import numpy as np
import pandas as pd
import cv2
from aicspylibczi import CziFile
import os
from scipy.spatial.distance import cdist
from scipy.spatial.distance import pdist, squareform
from tifffile import imwrite
from matplotlib import pyplot as plt

# Counters for skipped detections
skipped_by_area = 0
skipped_by_moments = 0
skipped_by_radius = 0
skipped_by_distance = 0
skipped_by_border = 0
skipped_by_visited = 0

# === USER PARAMETERS ===
folder_path = "/Users/millig/Documents/2509_autonomous_selection_pore_activity/Code"
dye_ch = 3                          # Dye channel index
guv_ch = 0                          # GUV channel index

# Physical and detection parameters
pixel_size_um = 638.9 / 512         # Microns per pixel
guv_diameter_range_um = (5, 100)    # Expected GUV diameter range in microns
inner_circle_fraction = 0.5         # Fraction of radius for inner intensity measurement
xy_tolerance_px = 12                # XY tolerance for grouping detections across Z
background_exclusion_scale = 1.0    # Scale for background exclusion radius

# Construct output CSV filename
last_two_parts = os.path.normpath(folder_path).split(os.sep)[-2:]
output_filename = f"{last_two_parts[0]}_{last_two_parts[1]}_summary.csv"
output_csv_path = os.path.join(folder_path, output_filename)

# Output directory for overlays
overlay_folder = os.path.join(folder_path, "overlays")
os.makedirs(overlay_folder, exist_ok=True)



# === FIND CZI Z-STACK FILES ===
def is_experiment_file(filename):
    """Identify valid experimental CZI z-stacks by filename."""
    return filename.startswith("Experiment") and filename.endswith(".czi")

input_files = [f for f in os.listdir(folder_path) if is_experiment_file(f)]

# Collect results for all files and timepoints
all_results = []

# === MAIN PROCESSING LOOP ===
for input_filename in input_files:
    input_path = os.path.join(folder_path, input_filename)
    print(f"\nProcessing: {input_filename}")

    # Load CZI file
    czi = CziFile(input_path)
    image_data = czi.read_image()
    img = np.squeeze(image_data[0])

    # Reshape depending on dimensionality
    if img.ndim >= 6:
        # Shape: (F, C, Z, T, H, W) → reorder to (C, F*T, Z, H, W)
        F, C, Z, T, H, W = img.shape
        img = np.transpose(img, (1, 0, 3, 2, 4, 5))
        img = img.reshape(C, F*T, Z, H, W)
    elif img.ndim == 5:
        # Shape: (C, Z, T, H, W) → (C, T, Z, H, W)
        C, Z, T, H, W = img.shape
        img = np.transpose(img, (0, 2, 1, 3, 4))

    print(f"Image shape: {img.shape}")

    # Extract channels
    C_dye_stack = img[dye_ch]
    C_GUV_stack = img[guv_ch]

    MAX_REGION_AREA = 0.5 * W * H
    print(f"C1 stack shape: {C_dye_stack.shape}, C2 stack shape: {C_GUV_stack.shape}")
    T, Z, H, W = C_GUV_stack.shape
    Z_total = Z

    # === PROCESS EACH TIMEPOINT ===
    for t in range(T):

        # Compute mean dye intensity per slice
        slice_means = [np.mean(C_dye_stack[t, z]) for z in range(Z)]
        threshold = np.percentile(slice_means, 0)
        valid_z_slices = list(range(Z))

        all_detections = []  # raw per-slice detections

        # === SLICE-BY-SLICE GUV DETECTION ===
        for z in range(Z_total):
            z_weight = z / (Z_total - 1)
            Color2 = C_GUV_stack[t, z]

            # Normalize + CLAHE
            Color2_norm = cv2.normalize(Color2, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            clahe = cv2.createCLAHE(clipLimit=5.0, tileGridSize=(8, 8))
            Color2_norm = clahe.apply(Color2_norm)

            # OPTIONAL: debugging
            # plt.imshow(Color2_norm, cmap='gray'); plt.title(f'z={z} CLAHE'); plt.show()

            # Gaussian smoothing
            Color2_norm = cv2.GaussianBlur(Color2_norm, (5, 5), 0)

            # Canny edge detection with z-dependent thresholding
            threshold1 = 50 - (z_weight * 10)
            factor = 3 - (z_weight)
            binary = cv2.Canny(Color2_norm, threshold1=threshold1, threshold2=factor * threshold1)

            # OPTIONAL: debugging
            # plt.imshow(binary, cmap='gray'); plt.title(f'z={z} binary'); plt.show()

            # Fill holes via flood fill
            holes_filled = binary.copy()
            mask = np.zeros((holes_filled.shape[0]+2, holes_filled.shape[1]+2), np.uint8)
            cv2.floodFill(holes_filled, mask, (0, 0), 255)
            holes_filled_inv = cv2.bitwise_not(holes_filled)
            binary_filled = binary | holes_filled_inv

            # Morphological opening
            kernel = np.ones((5, 5), np.uint8)
            binary_cleaned = cv2.morphologyEx(binary_filled, cv2.MORPH_OPEN, kernel)

            # Watershed segmentation prep
            dist_transform = cv2.distanceTransform(binary_cleaned, cv2.DIST_L2, 5)
            _, sure_fg = cv2.threshold(dist_transform, 0.3 * dist_transform.max(), 255, 0)
            sure_fg = np.uint8(sure_fg)
            unknown = cv2.subtract(binary_cleaned, sure_fg)

            num_labels, markers = cv2.connectedComponents(sure_fg)
            markers = markers + 1
            markers[unknown == 255] = 0

            markers_col = cv2.cvtColor(Color2_norm, cv2.COLOR_GRAY2BGR)
            markers_result = cv2.watershed(markers_col, markers)

            # Extract objects
            for label in np.unique(markers_result):
                if label <= 1:
                    continue

                mask_label = (markers_result == label).astype(np.uint8)
                num_pix = np.sum(mask_label)

                # Filter by area
                if num_pix < 10 or num_pix > MAX_REGION_AREA:
                    skipped_by_area += 1
                    continue

                # Compute centroid
                M = cv2.moments(mask_label)
                if M["m00"] == 0:
                    skipped_by_moments += 1
                    continue
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])

                # Radius estimate
                r_est = np.sqrt(num_pix / np.pi)
                r_um = r_est * pixel_size_um

                # Keep only radius in expected GUV range
                if guv_diameter_range_um[0]/2 <= r_um <= guv_diameter_range_um[1]/2:
                    all_detections.append({'x': cx, 'y': cy, 'r': int(r_est), 'z': z})

        # === GROUP DETECTIONS ACROSS Z (3D GUV reconstruction) ===
        groups = []
        visited = set()

        for i, seed in enumerate(all_detections):
            if i in visited:
                skipped_by_visited += 1
                continue
            group = [seed]
            visited.add(i)
            queue = [seed]

            while queue:
                current = queue.pop()
                for j, cand in enumerate(all_detections):
                    if j in visited:
                        continue
                    dz = abs(cand['z'] - current['z'])
                    if dz > 10:
                        continue
                    dist = np.linalg.norm([cand['x'] - current['x'], cand['y'] - current['y']])
                    if dist < xy_tolerance_px:
                        group.append(cand)
                        queue.append(cand)
                        visited.add(j)
            groups.append(group)

        # === FILTER OUT OVERLAPPING GUVs ===
        filtered_groups = []
        centers = []

        for group in groups:
            center = max(group, key=lambda g: g['r'])
            centers.append((center['x'], center['y'], center['r']))

        if centers:
            coords = np.array([[x, y] for x, y, r in centers])
            radii = np.array([r for x, y, r in centers])
            dists = squareform(pdist(coords))
            to_keep = np.ones(len(groups), dtype=bool)

            for i in range(len(groups)):
                if not to_keep[i]:
                    continue
                for j in range(i + 1, len(groups)):
                    if not to_keep[j]:
                        continue
                    min_dist = 0.5 * (radii[i] + radii[j])
                    if dists[i, j] < min_dist:
                        to_keep[j] = False

            groups = [g for i, g in enumerate(groups) if to_keep[i]]

        # === BACKGROUND MASK ===
        global_background_mask = np.ones_like(C_dye_stack[t], dtype=np.uint8) * 255
        for group in groups:
            center = max(group, key=lambda g: g['r'])
            zc, x, y, r = center['z'], center['x'], center['y'], center['r']
            if zc not in valid_z_slices:
                continue
            exclusion_radius = int(r * background_exclusion_scale)
            cv2.circle(global_background_mask[zc], (x, y), exclusion_radius, 0, -1)

        # Compute global background intensity
        valid_C1 = C_dye_stack[t][valid_z_slices]
        valid_mask = global_background_mask[valid_z_slices]
        background_pixels = valid_C1[valid_mask > 0]
        global_background_intensity = np.mean(background_pixels) if background_pixels.size > 0 else np.nan

        # === MEASURE INTENSITY PER GUV ===
        file_results = []
        for guv_id, group in enumerate(groups, 1):
            center = max(group, key=lambda g: g['r'])
            zc, x, y, r = center['z'], center['x'], center['y'], center['r']
            r_inner = int(r * inner_circle_fraction)

            if x - r_inner < 0 or x + r_inner >= W or y - r_inner < 0 or y + r_inner >= H:
                skipped_by_border += 1
                continue

            Color1 = C_dye_stack[t, zc]
            mask_inner = np.zeros_like(Color1, dtype=np.uint8)
            cv2.circle(mask_inner, (x, y), r_inner, 255, -1)
            mean_inner = cv2.mean(Color1, mask=mask_inner)[0]

            norm_intensity = mean_inner / global_background_intensity if global_background_intensity else np.nan

            file_results.append({
                'file_name': input_filename,
                'T': t,
                'GUV_ID': guv_id,
                'x': x, 'y': y, 'z': zc,
                'radius_px': r,
                'mean_C1_intensity': mean_inner,
                'mean_background_intensity': global_background_intensity,
                'normalized_inner_intensity': norm_intensity
            })

        # Add results globally
        all_results.extend(file_results)

        # === OVERLAY GENERATION ===
        C1_proj = cv2.normalize(np.max(C_dye_stack[t], axis=0), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        C2_proj = cv2.normalize(np.max(C_GUV_stack[t], axis=0), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

        overlay = np.zeros((H, W, 3), dtype=np.uint8)
        overlay[:, :, 0] = C2_proj
        overlay[:, :, 1] = C1_proj

        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        C2_proj_eq = clahe.apply(C2_proj)

        gamma = 0.8
        LUT = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)]).astype('uint8')
        C2_proj_bright = cv2.LUT(C2_proj_eq, LUT)
        overlay[:, :, 0] = C2_proj_bright

        # Draw circles
        for row in file_results:
            if row['T'] == t:
                cx, cy, r = row['x'], row['y'], row['radius_px']
                guv_id = row['GUV_ID']
                cv2.circle(overlay, (cx, cy), r, (255, 255, 255), 1)
                cv2.putText(overlay, str(guv_id), (cx - r, cy - r), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

        overlay_path = os.path.join(overlay_folder, f"{os.path.splitext(input_filename)[0]}_T{t}_overlay.png")
        cv2.imwrite(overlay_path, overlay)
        print(f"Saved overlay: {overlay_path}")

        # === TIFF Z-STACK OVERLAY ===
        overlay_stack = []
        for z in range(Z):
            C1_z = cv2.normalize(C_dye_stack[t, z], None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            C2_z = cv2.normalize(C_GUV_stack[t, z], None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

            ov = np.zeros((H, W, 3), dtype=np.uint8)
            ov[:, :, 0] = C2_z
            ov[:, :, 1] = C1_z

            C2_z_eq = clahe.apply(C2_z)
            C2_z_bright = cv2.LUT(C2_z_eq, LUT)
            ov[:, :, 0] = C2_z_bright

            for group in groups:
                for guv in group:
                    if guv['z'] == z:
                        cx, cy, r = guv['x'], guv['y'], guv['r']
                        guv_id = groups.index(group) + 1
                        cv2.circle(ov, (cx, cy), r, (255, 255, 255), 1)
                        cv2.putText(ov, str(guv_id), (cx - r, cy - r), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

            overlay_stack.append(ov)

        overlay_stack = np.stack(overlay_stack, axis=0)
        tiff_path = os.path.join(overlay_folder, f"{os.path.splitext(input_filename)[0]}_T{t}_overlay_stack.tiff")
        imwrite(tiff_path, overlay_stack)
        print(f"Saved overlay TIFF stack: {tiff_path}")

# === SAVE RESULTS ===
df = pd.DataFrame(all_results)
df.to_csv(output_csv_path, index=False)

print("Skipped by area:", skipped_by_area)
print("Skipped by radius:", skipped_by_radius)
print("Skipped by moments:", skipped_by_moments)
print("Skipped by distance:", skipped_by_distance)
print("Skipped by border:", skipped_by_border)
print("Skipped by visited:", skipped_by_visited)
print("Total detections:", len(all_results))

print(f"\n✓ All done. Results saved to: {output_csv_path}")



Processing: Experiment_demo.czi
Image shape: (4, 8, 80, 512, 512)
C1 stack shape: (8, 80, 512, 512), C2 stack shape: (8, 80, 512, 512)
Saved overlay: /Users/millig/Documents/2509_autonomous_selection_pore_activity/Code/overlays/Experiment_demo_T0_overlay.png
Saved overlay TIFF stack: /Users/millig/Documents/2509_autonomous_selection_pore_activity/Code/overlays/Experiment_demo_T0_overlay_stack.tiff
Saved overlay: /Users/millig/Documents/2509_autonomous_selection_pore_activity/Code/overlays/Experiment_demo_T1_overlay.png
Saved overlay TIFF stack: /Users/millig/Documents/2509_autonomous_selection_pore_activity/Code/overlays/Experiment_demo_T1_overlay_stack.tiff
Saved overlay: /Users/millig/Documents/2509_autonomous_selection_pore_activity/Code/overlays/Experiment_demo_T2_overlay.png
Saved overlay TIFF stack: /Users/millig/Documents/2509_autonomous_selection_pore_activity/Code/overlays/Experiment_demo_T2_overlay_stack.tiff
Saved overlay: /Users/millig/Documents/2509_autonomous_selection_p